# NBA Injury/Transaction Scraper

Scrapes injury, personal, and disciplinary data from prosportstransactions.com

**Features:**
- Resume capability: automatically skips already-scraped teams on restart
- Checkpoint saves: data saved after each team in case of crash
- Conservative rate limiting: 3-5 second delays between requests
- Retry logic: exponential backoff for failed requests

---

## 1. Setup & Imports

In [1]:
# Install dependencies if needed (uncomment to run)
# !pip install requests beautifulsoup4 pandas python-dotenv sqlalchemy psycopg2-binary

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import json
from datetime import datetime
from dotenv import load_dotenv
from sqlalchemy import create_engine

print("Imports successful!")

Imports successful!


## 2. Configuration

**Edit these values to customize your scrape:**

In [3]:
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES
# =============================================================================

START_DATE = "2009-01-01"  # Format: YYYY-MM-DD
END_DATE = "2026-01-07"    # Format: YYYY-MM-DD

# Conservative delay range (seconds) between requests
MIN_DELAY = 3
MAX_DELAY = 5

# Checkpoint save frequency (save after every N teams)
CHECKPOINT_FREQUENCY = 1  # Save after each team

# Output files
CHECKPOINT_FILE = "injury_data_checkpoint.csv"
FINAL_FILE = "injury_data_final.csv"
PROGRESS_FILE = "scrape_progress.json"  # Tracks completed teams for resume

# Database table name
DB_TABLE_NAME = "nba_injury_transactions"

# Resume behavior
RESUME_ENABLED = True  # Set to False to force fresh start (ignores progress file)

print(f"Configuration loaded:")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Delay: {MIN_DELAY}-{MAX_DELAY} seconds")
print(f"  Resume enabled: {RESUME_ENABLED}")

Configuration loaded:
  Date range: 2009-01-01 to 2026-01-07
  Delay: 3-5 seconds
  Resume enabled: True


## 3. Team Dictionary

All 30 NBA teams plus historical names (Bobcats, New Jersey Nets, etc.)

In [4]:
NBA_TEAMS = {
    # Eastern Conference - Atlantic
    "Boston Celtics": "Celtics",
    "Brooklyn Nets": "Nets",
    "New Jersey Nets": "Nets",  # Historical (pre-2012)
    "New York Knicks": "Knicks",
    "Philadelphia 76ers": "76ers",
    "Toronto Raptors": "Raptors",
    
    # Eastern Conference - Central
    "Chicago Bulls": "Bulls",
    "Cleveland Cavaliers": "Cavaliers",
    "Detroit Pistons": "Pistons",
    "Indiana Pacers": "Pacers",
    "Milwaukee Bucks": "Bucks",
    
    # Eastern Conference - Southeast
    "Atlanta Hawks": "Hawks",
    "Charlotte Hornets": "Hornets",
    "Charlotte Bobcats": "Bobcats",  # Historical (2004-2014)
    "Miami Heat": "Heat",
    "Orlando Magic": "Magic",
    "Washington Wizards": "Wizards",
    
    # Western Conference - Northwest
    "Denver Nuggets": "Nuggets",
    "Minnesota Timberwolves": "Timberwolves",
    "Oklahoma City Thunder": "Thunder",
    "Portland Trail Blazers": "Trail Blazers",
    "Utah Jazz": "Jazz",
    
    # Western Conference - Pacific
    "Golden State Warriors": "Warriors",
    "Los Angeles Clippers": "Clippers",
    "Los Angeles Lakers": "Lakers",
    "Phoenix Suns": "Suns",
    "Sacramento Kings": "Kings",
    
    # Western Conference - Southwest
    "Dallas Mavericks": "Mavericks",
    "Houston Rockets": "Rockets",
    "Memphis Grizzlies": "Grizzlies",
    "New Orleans Pelicans": "Pelicans",
    "New Orleans Hornets": "Hornets",  # Historical (2002-2013)
    "San Antonio Spurs": "Spurs",
}

# Deduplicated list for scraping
TEAMS_TO_SCRAPE = sorted(list(set(NBA_TEAMS.values())))

print(f"Teams to scrape ({len(TEAMS_TO_SCRAPE)}):")
for i, team in enumerate(TEAMS_TO_SCRAPE, 1):
    print(f"  {i:2}. {team}")

Teams to scrape (31):
   1. 76ers
   2. Bobcats
   3. Bucks
   4. Bulls
   5. Cavaliers
   6. Celtics
   7. Clippers
   8. Grizzlies
   9. Hawks
  10. Heat
  11. Hornets
  12. Jazz
  13. Kings
  14. Knicks
  15. Lakers
  16. Magic
  17. Mavericks
  18. Nets
  19. Nuggets
  20. Pacers
  21. Pelicans
  22. Pistons
  23. Raptors
  24. Rockets
  25. Spurs
  26. Suns
  27. Thunder
  28. Timberwolves
  29. Trail Blazers
  30. Warriors
  31. Wizards


## 4. Scraper Class Definition

In [5]:
class NBAInjuryScraper:
    BASE_URL = "https://prosportstransactions.com/basketball/Search/SearchResults.php"
    
    def __init__(self, start_date: str, end_date: str):
        self.start_date = start_date
        self.end_date = end_date
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Connection': 'keep-alive',
        })
        self.all_data = []
        self.completed_teams = set()
        self._load_progress()
        
    def _load_progress(self):
        """Load progress from previous run if resume is enabled."""
        if not RESUME_ENABLED:
            print("[RESUME] Resume disabled - starting fresh")
            self.completed_teams = set()
            return
            
        if os.path.exists(PROGRESS_FILE):
            try:
                with open(PROGRESS_FILE, 'r') as f:
                    progress_data = json.load(f)
                    
                # Verify date range matches
                if (progress_data.get('start_date') == self.start_date and 
                    progress_data.get('end_date') == self.end_date):
                    self.completed_teams = set(progress_data.get('completed_teams', []))
                    print(f"[RESUME] Loaded progress: {len(self.completed_teams)} teams already completed")
                    if self.completed_teams:
                        print(f"[RESUME] Completed teams: {sorted(self.completed_teams)}")
                else:
                    print("[RESUME] Date range changed - starting fresh scrape")
                    self.completed_teams = set()
            except (json.JSONDecodeError, KeyError) as e:
                print(f"[RESUME] Could not load progress file: {e}")
                self.completed_teams = set()
        else:
            print("[RESUME] No progress file found - starting fresh")
            self.completed_teams = set()
            
        # Load existing checkpoint data if resuming
        if self.completed_teams and os.path.exists(CHECKPOINT_FILE):
            try:
                existing_df = pd.read_csv(CHECKPOINT_FILE)
                self.all_data = existing_df.to_dict('records')
                print(f"[RESUME] Loaded {len(self.all_data)} existing rows from checkpoint")
            except Exception as e:
                print(f"[RESUME] Could not load checkpoint data: {e}")
                self.all_data = []
    
    def _save_progress(self):
        """Save current progress to file."""
        progress_data = {
            'start_date': self.start_date,
            'end_date': self.end_date,
            'completed_teams': list(self.completed_teams),
            'last_updated': datetime.now().isoformat(),
            'total_rows': len(self.all_data),
        }
        with open(PROGRESS_FILE, 'w') as f:
            json.dump(progress_data, f, indent=2)
    
    def _mark_team_completed(self, team: str):
        """Mark a team as completed and save progress."""
        self.completed_teams.add(team)
        self._save_progress()
        
    def _build_url(self, team: str, start: int = 0) -> str:
        """Build the URL for a specific team and pagination offset."""
        params = {
            'Player': '',
            'Team': team,
            'BeginDate': self.start_date,
            'EndDate': self.end_date,
            'InjuriesChkBx': 'yes',
            'PersonalChkBx': 'yes',
            'DisciplinaryChkBx': 'yes',
            'Submit': 'Search',
        }
        if start > 0:
            params['start'] = start
            
        param_str = '&'.join([f"{k}={v}" for k, v in params.items()])
        return f"{self.BASE_URL}?{param_str}"
    
    def _random_delay(self):
        """Sleep for a random duration within the configured range."""
        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        time.sleep(delay)
        
    def _fetch_page(self, url: str, retries: int = 3):
        """Fetch a page with retry logic and exponential backoff."""
        for attempt in range(retries):
            try:
                response = self.session.get(url, timeout=30)
                response.raise_for_status()
                return BeautifulSoup(response.text, 'html.parser')
            except requests.RequestException as e:
                print(f"    [ERROR] Request failed (attempt {attempt + 1}/{retries}): {e}")
                if attempt < retries - 1:
                    wait_time = (2 ** attempt) * 5
                    print(f"    [RETRY] Waiting {wait_time} seconds before retry...")
                    time.sleep(wait_time)
                else:
                    print(f"    [FAILED] All retries exhausted for URL: {url}")
                    return None
        return None
    
    def _get_total_pages(self, soup) -> int:
        """Parse the pagination section to determine total number of pages."""
        paging_tables = soup.find_all('table', {'width': '75%'})
        
        for table in paging_tables:
            links = table.find_all('a')
            page_numbers = []
            
            for link in links:
                text = link.get_text(strip=True)
                if text.lower() in ['previous', 'next']:
                    continue
                try:
                    page_num = int(text)
                    page_numbers.append(page_num)
                except ValueError:
                    continue
            
            if page_numbers:
                return max(page_numbers)
        
        return 1
    
    def _parse_table(self, soup, team: str) -> list:
        """Parse the data table from the page."""
        rows = []
        
        table = soup.find('table', class_='datatable')
        if not table:
            return rows
            
        tr_elements = table.find_all('tr')
        
        for tr in tr_elements:
            if tr.find('td', class_='DraftTableLabel') or tr.find('tr', class_='DraftTableLabel'):
                continue
            if 'DraftTableLabel' in tr.get('class', []):
                continue
                
            tds = tr.find_all('td')
            if len(tds) >= 5:
                date_text = tds[0].get_text(strip=True)
                team_text = tds[1].get_text(strip=True)
                acquired_text = tds[2].get_text(strip=True)
                relinquished_text = tds[3].get_text(strip=True)
                notes_text = tds[4].get_text(strip=True)
                
                if date_text.lower() == 'date':
                    continue
                
                if date_text:
                    rows.append({
                        'date': date_text,
                        'team': team_text if team_text else team,
                        'acquired': acquired_text,
                        'relinquished': relinquished_text,
                        'notes': notes_text,
                        'scrape_timestamp': datetime.now().isoformat(),
                    })
        
        return rows
    
    def _check_for_no_results(self, soup) -> bool:
        """Check if the page indicates no results found."""
        page_text = soup.get_text()
        if "No records found" in page_text or "0 records" in page_text:
            return True
        table = soup.find('table', class_='datatable')
        if not table:
            return True
        rows = table.find_all('tr')
        data_rows = [r for r in rows if not r.find('td', class_='DraftTableLabel') 
                     and 'DraftTableLabel' not in r.get('class', [])]
        return len(data_rows) <= 1
    
    def scrape_team(self, team: str) -> list:
        """Scrape all pages for a single team."""
        print(f"\n[TEAM] Scraping: {team}")
        team_data = []
        
        url = self._build_url(team, start=0)
        print(f"  [PAGE 1] Fetching: {url[:80]}...")
        
        soup = self._fetch_page(url)
        if not soup:
            print(f"  [ERROR] Failed to fetch first page for {team}")
            return team_data
        
        if self._check_for_no_results(soup):
            print(f"  [INFO] No results found for {team}")
            return team_data
        
        total_pages = self._get_total_pages(soup)
        print(f"  [INFO] Total pages detected: {total_pages}")
        
        page_rows = self._parse_table(soup, team)
        team_data.extend(page_rows)
        print(f"  [PAGE 1] Extracted {len(page_rows)} rows")
        
        for page_num in range(2, total_pages + 1):
            self._random_delay()
            
            start_offset = (page_num - 1) * 25
            url = self._build_url(team, start=start_offset)
            
            print(f"  [PAGE {page_num}/{total_pages}] Fetching...")
            
            soup = self._fetch_page(url)
            if not soup:
                print(f"  [ERROR] Failed to fetch page {page_num}")
                continue
                
            page_rows = self._parse_table(soup, team)
            team_data.extend(page_rows)
            print(f"  [PAGE {page_num}] Extracted {len(page_rows)} rows")
            
            if len(page_rows) == 0:
                print(f"  [INFO] No more data, stopping pagination")
                break
        
        print(f"  [TEAM COMPLETE] {team}: {len(team_data)} total rows")
        return team_data
    
    def scrape_all_teams(self, teams: list = None):
        """Scrape all teams (or a specified subset)."""
        if teams is None:
            teams = TEAMS_TO_SCRAPE
        
        teams_to_process = [t for t in teams if t not in self.completed_teams]
        skipped_count = len(teams) - len(teams_to_process)
            
        print("=" * 60)
        print("NBA INJURY TRANSACTION SCRAPER")
        print("=" * 60)
        print(f"Date Range: {self.start_date} to {self.end_date}")
        print(f"Total teams: {len(teams)}")
        if skipped_count > 0:
            print(f"Already completed (skipping): {skipped_count}")
        print(f"Teams to scrape this run: {len(teams_to_process)}")
        print(f"Delay between requests: {MIN_DELAY}-{MAX_DELAY} seconds")
        print(f"Resume enabled: {RESUME_ENABLED}")
        print("=" * 60)
        
        if not teams_to_process:
            print("\n[INFO] All teams already scraped! Nothing to do.")
            print("[INFO] To force a fresh scrape, set RESUME_ENABLED = False or run reset_progress()")
            return self.get_dataframe()
        
        for idx, team in enumerate(teams_to_process, 1):
            print(f"\n[PROGRESS] Team {idx}/{len(teams_to_process)} (Overall: {len(self.completed_teams) + idx}/{len(teams)})")
            
            team_data = self.scrape_team(team)
            self.all_data.extend(team_data)
            
            self._mark_team_completed(team)
            
            if idx % CHECKPOINT_FREQUENCY == 0:
                self._save_checkpoint()
            
            if idx < len(teams_to_process):
                self._random_delay()
        
        print("\n" + "=" * 60)
        print("SCRAPING COMPLETE")
        print(f"Total rows collected: {len(self.all_data)}")
        print(f"Teams completed: {len(self.completed_teams)}")
        print("=" * 60)
        
        return self.get_dataframe()
    
    def _save_checkpoint(self):
        """Save current data to checkpoint file."""
        if self.all_data:
            df = pd.DataFrame(self.all_data)
            df.to_csv(CHECKPOINT_FILE, index=False)
            print(f"  [CHECKPOINT] Saved {len(self.all_data)} rows to {CHECKPOINT_FILE}")
    
    def get_dataframe(self) -> pd.DataFrame:
        """Return collected data as a DataFrame."""
        if not self.all_data:
            return pd.DataFrame()
        return pd.DataFrame(self.all_data)
    
    def save_to_csv(self, filename: str = None):
        """Save data to CSV file."""
        if filename is None:
            filename = FINAL_FILE
        df = self.get_dataframe()
        if not df.empty:
            df.to_csv(filename, index=False)
            print(f"[SAVED] {len(df)} rows to {filename}")
        else:
            print("[WARNING] No data to save")
        return df

print("NBAInjuryScraper class defined!")

NBAInjuryScraper class defined!


## 5. Helper Functions

In [6]:
def save_to_database(df: pd.DataFrame, table_name: str = DB_TABLE_NAME):
    """Save DataFrame to Supabase database."""
    print("\n[DATABASE] Connecting to database...")
    
    load_dotenv()
    database_url = os.getenv("DATABASE_URL")
    
    if not database_url:
        print("[ERROR] DATABASE_URL not found in environment variables")
        print("[INFO] Data saved to CSV only. Set DATABASE_URL to enable database saving.")
        return False
    
    try:
        engine = create_engine(database_url)
        
        # Clean the dataframe for postgres
        df_clean = df.copy()
        
        # Convert date column to proper datetime
        df_clean['date'] = pd.to_datetime(df_clean['date'], errors='coerce')
        
        # Replace NaN with None for proper NULL handling
        df_clean = df_clean.where(pd.notnull(df_clean), None)
        
        # Replace empty strings with None
        df_clean = df_clean.replace('', None)
        
        # Clean the bullet point characters from player names
        if 'acquired' in df_clean.columns:
            df_clean['acquired'] = df_clean['acquired'].str.replace('•', '').str.strip()
        if 'relinquished' in df_clean.columns:
            df_clean['relinquished'] = df_clean['relinquished'].str.replace('•', '').str.strip()
        
        print(f"[DATABASE] Saving {len(df_clean)} rows to table '{table_name}'...")
        
        df_clean.to_sql(
            table_name,
            engine,
            if_exists='append',  # Use 'replace' if you want to overwrite
            index=False
        )
        
        print(f"[DATABASE] Successfully saved to '{table_name}'")
        
        # Verify the save
        row_count = pd.read_sql(f"SELECT COUNT(*) as count FROM {table_name}", engine)
        print(f"[DATABASE] Total rows in table: {row_count['count'].iloc[0]}")
        
        return True
        
    except Exception as e:
        print(f"[ERROR] Database save failed: {e}")
        print("[INFO] Data preserved in CSV checkpoint file")
        return False


def reset_progress():
    """Delete progress and checkpoint files to start fresh."""
    files_to_delete = [PROGRESS_FILE, CHECKPOINT_FILE]
    for f in files_to_delete:
        if os.path.exists(f):
            os.remove(f)
            print(f"[RESET] Deleted {f}")
        else:
            print(f"[RESET] {f} does not exist")
    print("[RESET] Progress reset complete - next run will start fresh")


def show_progress():
    """Display current scraping progress."""
    print("=" * 60)
    print("SCRAPING PROGRESS STATUS")
    print("=" * 60)
    
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            progress = json.load(f)
        
        completed = progress.get('completed_teams', [])
        total_teams = len(TEAMS_TO_SCRAPE)
        
        print(f"Date Range: {progress.get('start_date')} to {progress.get('end_date')}")
        print(f"Last Updated: {progress.get('last_updated')}")
        print(f"Teams Completed: {len(completed)}/{total_teams}")
        print(f"Total Rows Collected: {progress.get('total_rows', 0)}")
        
        remaining = [t for t in TEAMS_TO_SCRAPE if t not in completed]
        if remaining:
            print(f"\nRemaining teams ({len(remaining)}):")
            for t in sorted(remaining):
                print(f"  - {t}")
        else:
            print("\nAll teams completed!")
    else:
        print("No progress file found - scraping has not started yet")
    
    print("=" * 60)

print("Helper functions defined!")

Helper functions defined!


## 6. Check Current Progress

Run this cell to see the current status of any previous scraping runs:

In [7]:
show_progress()

SCRAPING PROGRESS STATUS
No progress file found - scraping has not started yet


## 7. Reset Progress (Optional)

**Only run this if you want to start completely fresh:**

In [8]:
# UNCOMMENT THE LINE BELOW TO RESET PROGRESS
# reset_progress()

## 8. Run the Scraper

This will scrape all teams (or resume from where you left off).

**Estimated time:** 30-60+ minutes for full scrape (2009-2026) with conservative rate limiting.

In [9]:
# Initialize the scraper
scraper = NBAInjuryScraper(START_DATE, END_DATE)

[RESUME] No progress file found - starting fresh


In [10]:
# Run the scraper (will resume if previous progress exists)
df = scraper.scrape_all_teams()

NBA INJURY TRANSACTION SCRAPER
Date Range: 2009-01-01 to 2026-01-07
Total teams: 31
Teams to scrape this run: 31
Delay between requests: 3-5 seconds
Resume enabled: True

[PROGRESS] Team 1/31 (Overall: 1/31)

[TEAM] Scraping: 76ers
  [PAGE 1] Fetching: https://prosportstransactions.com/basketball/Search/SearchResults.php?Player=&Te...
    [ERROR] Request failed (attempt 1/3): 403 Client Error: Forbidden for url: https://prosportstransactions.com/basketball/Search/SearchResults.php?Player=&Team=76ers&BeginDate=2009-01-01&EndDate=2026-01-07&InjuriesChkBx=yes&PersonalChkBx=yes&DisciplinaryChkBx=yes&Submit=Search
    [RETRY] Waiting 5 seconds before retry...
    [ERROR] Request failed (attempt 2/3): 403 Client Error: Forbidden for url: https://prosportstransactions.com/basketball/Search/SearchResults.php?Player=&Team=76ers&BeginDate=2009-01-01&EndDate=2026-01-07&InjuriesChkBx=yes&PersonalChkBx=yes&DisciplinaryChkBx=yes&Submit=Search
    [RETRY] Waiting 10 seconds before retry...
    [ERROR

KeyboardInterrupt: 

In [ ]:
# Save to CSV
scraper.save_to_csv()

## 9. Preview the Data

In [ ]:
# Show basic info
print(f"Total rows: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Show first 20 rows
df.head(20)

In [ ]:
# Rows per team
df['team'].value_counts()

## 10. Save to Database

Make sure your `.env` file contains `DATABASE_URL` before running this cell.

In [ ]:
# Save to Supabase database
if not df.empty:
    save_to_database(df)
else:
    print("No data to save to database")

## 11. Scrape Specific Teams Only (Optional)

If you only want to scrape specific teams, use this cell instead:

In [ ]:
# Example: Scrape only specific teams
# specific_teams = ["Pistons", "Lakers", "Celtics"]
# scraper = NBAInjuryScraper(START_DATE, END_DATE)
# df = scraper.scrape_all_teams(teams=specific_teams)

## 12. Load Existing Data (Optional)

If you already have scraped data saved and just want to load it:

In [ ]:
# Load from checkpoint file
# df = pd.read_csv(CHECKPOINT_FILE)
# print(f"Loaded {len(df)} rows from {CHECKPOINT_FILE}")
# df.head()

In [ ]:
# Load from final file
# df = pd.read_csv(FINAL_FILE)
# print(f"Loaded {len(df)} rows from {FINAL_FILE}")
# df.head()